# Curve stacker: Branch A + p(AI) curves

The own approach. A binary human-vs-ai window detector (Branch B, trained on the detector set: human and ai texts of the fixed 50k sample) scores every window of every text, giving one p(AI) curve per text and window size (64, 128, 256 tokens). Curve features plus Branch A's probabilities (the tuned 3-class XLM-R, `hw3-encoder-tuning.ipynb`) go into a stacker that predicts human / ai / fusion.

- **Fitting**: the stacker, its scaler and the class weights are fitted on the **stacker set** only (30k training texts the detector and Branch A never saw, twin-guarded; see `curves_check.ipynb`, section 1). Model choice uses 5-fold CV on the stacker set.
- **Evaluation**: the fixed validation fold (fold 0), scored once per configuration, against the reference R0 = Branch A, mean over windows (val macro-F1 0.902).
- Model scoring ran on the cluster (`make -C cluster full`); this notebook runs on CPU.

In [2]:
import sys
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import ParameterGrid, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

sys.path.insert(0, str(Path("../cluster").resolve()))
from encoder.curve_features import (BASIC, CHANGEPOINT, CONTEXT, SEGMENT, cross_scale_features, curve_features,
                                    subsample)
from encoder.data import LABELS, SEED, load_sets

COMPETITION = "ai-generated-text-detection-hw3-ucu-ai-2026"
FULL_JOB = "3016350"                                     # job id of `make -C cluster full`
RESULTS_DIR = Path("../cluster/results") / FULL_JOB
KAGGLE_PROBS = Path("../outputs/branch_a/encoder")    # Branch A probabilities from the Kaggle run
FEATURE_DIR = Path("../outputs/curves")               # cached curve features (recomputing takes minutes)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

SETS = ["stacker", "val", "test"]
SIZES = [64, 128, 256]
STRIDES = {"fine": 1, "coarse": 2}   # subsample factor of the size/4 window grid: stride size/4 and size/2
N_PARTS = 4                          # one part file per GPU

sets = load_sets(kagglehub.competition_download(COMPETITION))
pd.DataFrame({"texts": {name: len(df) for name, df in sets.items()}}).T

,detector,stacker,val,test
texts,34638,29999,27236,25000


## 1. Cluster outputs
Branch A was scored again on the cluster for all three sets, so the stacker's training and evaluation inputs come from the same hardware and precision. The first check compares the cluster's validation probabilities with the Kaggle run's (fp16 on T4 vs bf16 on GH200): they should agree almost exactly, and give the same R0.

In [3]:
def read_parts(pattern):
    """All per-GPU part files of one cluster output, concatenated."""
    parts = sorted(RESULTS_DIR.glob(pattern))
    assert len(parts) == N_PARTS, f"{pattern}: {len(parts)} part files"
    return pd.concat([pd.read_csv(path) for path in parts], ignore_index=True)


def branch_a_probs(set_name):
    """Branch A's text probabilities of one set from the cluster, in the set's row order."""
    probs = read_parts(f"branch_a_probs_{set_name}.part*.csv").set_index("id")[LABELS]
    return probs.loc[sets[set_name]["id"]]


def score(y_true, y_pred):
    """Macro-F1 and per-class F1."""
    per_class = f1_score(y_true, y_pred, labels=LABELS, average=None)
    return {"macro_f1": f1_score(y_true, y_pred, average="macro"),
            **{f"f1_{label}": f1 for label, f1 in zip(LABELS, per_class)}}


def to_labels(probs):
    """Class name of the most probable class per row."""
    return np.array(LABELS)[np.asarray(probs).argmax(axis=1)]


probs_a = {set_name: branch_a_probs(set_name) for set_name in SETS}
kaggle_val = pd.read_csv(KAGGLE_PROBS / "val_probs_xlm-roberta-base.csv").set_index("id").loc[probs_a["val"].index]
y = {set_name: sets[set_name]["label"].to_numpy() for set_name in ("stacker", "val")}
pd.DataFrame({
    "Kaggle run (R0)": score(y["val"], to_labels(kaggle_val)),
    "cluster run": score(y["val"], to_labels(probs_a["val"])),
}).T.assign(
    mean_abs_prob_diff=[0.0, (probs_a["val"] - kaggle_val).abs().to_numpy().mean()],
    same_argmax_pct=[100.0, 100 * (to_labels(probs_a["val"]) == to_labels(kaggle_val)).mean()],
).round(4)

,macro_f1,f1_human,f1_ai,f1_fusion,mean_abs_prob_diff,same_argmax_pct
Kaggle run (R0),0.9020,0.8362,0.9901,0.8797,0.0000,100.0000
cluster run,0.9023,0.8368,0.9901,0.8801,0.0006,99.8899


## 2. Curve features
For every set, window size and stride, the features of `encoder/curve_features.py`, one row per text:
- **basic**: mean, max, std of p(AI), share of high windows (> 0.5), longest high run as a share of the text, mean jump between neighbours, number of low/high switches;
- **changepoint**: the best single step up (gain and position), head and tail means, the last window;
- **segment**: the most AI-like contiguous stretch (start, end, length share, mean, contrast with the rest). The gate showed the rewritten part is often a stretch in the middle, not a suffix;
- **context**: number of windows and log length, so the stacker knows when a curve is too short to trust;
- **cross-scale** (fine stride only): agreement of neighbouring window sizes (64–128, 128–256): curve correlation, gap between maxima, overlap of high regions.

Strides: *fine* = a window every size/4 tokens (as scored), *coarse* = every size/2 tokens (every other window). Columns are named `<size>_<stride>__<feature>` and `cross_<a>_<b>_<feature>`. Features are cached as CSV in `outputs/curves/`.

In [4]:
def set_features(set_name):
    """All curve features of one set (every size and stride, plus cross-scale), cached as CSV."""
    path = FEATURE_DIR / f"features_{FULL_JOB}_{set_name}.csv"
    if path.exists():
        return pd.read_csv(path, index_col="id")
    curves = {size: read_parts(f"curves_{set_name}_{size}.part*.csv.gz") for size in SIZES}
    parts = [curve_features(subsample(curves[size], factor)).add_prefix(f"{size}_{stride}__")
             for size in SIZES for stride, factor in STRIDES.items()]
    parts += [cross_scale_features(curves[a], curves[b], f"cross_{a}_{b}") for a, b in zip(SIZES, SIZES[1:])]
    features = pd.concat(parts, axis=1)
    features.index.name = "id"
    features.to_csv(path)
    return features


def with_branch_a(features, probs):
    """Curve features joined with Branch A's log-probabilities (columns logp_<class>)."""
    log_probs = np.log(probs.clip(lower=1e-6)).add_prefix("logp_")
    return log_probs.join(features)


X = {set_name: with_branch_a(set_features(set_name), probs_a[set_name]).loc[sets[set_name]["id"]]
     for set_name in tqdm(SETS, desc="features")}
assert all(not X[s].isna().any().any() for s in SETS), "missing feature values"
pd.DataFrame({"texts": {s: len(X[s]) for s in SETS}, "columns": {s: X[s].shape[1] for s in SETS}}).T

features:   0%|          | 0/3 [00:00<?, ?it/s]

,stacker,val,test
texts,29999,27236,25000
columns,129,129,129


## 3. Experiments
Each configuration is a **feature set** (which inputs the stacker sees) × a **model** with its parameters:
- **R0** Branch A, argmax (no fitting): the fixed reference.
- **R1** Branch A + class weights: Branch A's probabilities multiplied by per-class weights tuned for macro-F1 on the stacker set. How much is thresholding alone?
- **R2** stacker on Branch A only: is any gain just from having a fitted stacker?
- **E1** curves only (all sizes); **E2** Branch A + curves (all sizes): the main result.
- **E3** window sizes: Branch A + one size, or two sizes.
- **E4** stride: Branch A + all sizes, coarse stride (size/2) instead of fine (size/4).
- **E5** feature groups: basic → + changepoint → + segment → + cross-scale.

For each feature set, the model parameters are chosen by 5-fold CV macro-F1 on the stacker set; the class weights are then tuned on the out-of-fold probabilities; the final model is refitted on the whole stacker set and scored once on validation. The interval is a paired bootstrap (1,000 resamples of the validation texts) of the macro-F1 difference to R0.

In [5]:
BRANCH_A = [f"logp_{label}" for label in LABELS]
GROUPS = {"basic": BASIC, "changepoint": CHANGEPOINT, "segment": SEGMENT, "context": CONTEXT}
ALL_GROUPS = ["basic", "changepoint", "segment", "context"]

FEATURE_SETS = {
    "R2: Branch A":                       {"branch_a": True,  "sizes": [],             "groups": [],          "cross": False},
    "E1: curves only":                    {"branch_a": False, "sizes": SIZES,          "groups": ALL_GROUPS,  "cross": True},
    "E2: Branch A + curves":              {"branch_a": True,  "sizes": SIZES,          "groups": ALL_GROUPS,  "cross": True},
    "E3: Branch A + 64":                  {"branch_a": True,  "sizes": [64],           "groups": ALL_GROUPS,  "cross": False},
    "E3: Branch A + 128":                 {"branch_a": True,  "sizes": [128],          "groups": ALL_GROUPS,  "cross": False},
    "E3: Branch A + 256":                 {"branch_a": True,  "sizes": [256],          "groups": ALL_GROUPS,  "cross": False},
    "E3: Branch A + 128, 256":            {"branch_a": True,  "sizes": [128, 256],     "groups": ALL_GROUPS,  "cross": True},
    "E4: Branch A + curves, coarse":      {"branch_a": True,  "sizes": SIZES,          "groups": ALL_GROUPS,  "cross": True,
                                           "stride": "coarse"},
    "E5: Branch A + basic":               {"branch_a": True,  "sizes": SIZES,          "groups": ["basic", "context"], "cross": False},
    "E5: + changepoint":                  {"branch_a": True,  "sizes": SIZES,          "groups": ["basic", "changepoint", "context"],
                                           "cross": False},
    "E5: + changepoint + segment":        {"branch_a": True,  "sizes": SIZES,          "groups": ALL_GROUPS,  "cross": False},
}
MODELS = {"LogReg": LogisticRegression, "HGB": HistGradientBoostingClassifier}
GRID = {
    "LogReg": {"C": [0.1, 1.0, 10.0], "class_weight": [None, "balanced"], "max_iter": [5_000]},
    "HGB": {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [15, 31], "max_iter": [300], "class_weight": [None, "balanced"],
            "random_state": [SEED]},
}
GBM_FEATURE_SETS = ["R2: Branch A", "E2: Branch A + curves"]  # HGB is slower: only for the main comparison
CV_FOLDS = 5


def columns_for(spec, columns):
    """Feature columns of one feature-set spec."""
    stride = spec.get("stride", "fine")
    chosen = list(BRANCH_A) if spec["branch_a"] else []
    names = [name for group in spec["groups"] for name in GROUPS[group]]
    chosen += [f"{size}_{stride}__{name}" for size in spec["sizes"] for name in names]
    if spec["cross"]:
        pairs = [(a, b) for a, b in zip(SIZES, SIZES[1:]) if a in spec["sizes"] and b in spec["sizes"]]
        chosen += [c for a, b in pairs for c in columns if c.startswith(f"cross_{a}_{b}_")]
    missing = set(chosen) - set(columns)
    assert not missing, f"missing columns: {sorted(missing)[:5]}"
    return chosen


def make_model(model_name, params):
    """A fresh model; logistic regression gets a scaler fitted inside the same pipeline."""
    model = MODELS[model_name](**params)
    return make_pipeline(StandardScaler(), model) if model_name == "LogReg" else model


configs = [(feature_set, model_name, params) for feature_set in FEATURE_SETS for model_name in MODELS
           for params in ParameterGrid(GRID[model_name])
           if model_name == "LogReg" or feature_set in GBM_FEATURE_SETS]
print(f"{len(FEATURE_SETS)} feature sets, {len(configs)} configurations, {CV_FOLDS}-fold CV on the stacker set")

11 feature sets, 82 configurations, 5-fold CV on the stacker set


In [6]:
def oof_probs(feature_set, model_name, params):
    """Out-of-fold class probabilities on the stacker set (columns in LABELS order)."""
    columns = columns_for(FEATURE_SETS[feature_set], X["stacker"].columns)
    cv = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
    probs = cross_val_predict(make_model(model_name, params), X["stacker"][columns], y["stacker"], cv=cv,
                              method="predict_proba", n_jobs=CV_FOLDS)
    classes = sorted(LABELS)  # sklearn orders classes alphabetically
    return pd.DataFrame(probs, columns=classes)[LABELS].to_numpy()


def run_cv(config):
    """CV macro-F1 (argmax of out-of-fold probabilities) of one configuration, with its OOF probabilities."""
    feature_set, model_name, params = config
    probs = oof_probs(feature_set, model_name, params)
    return {"feature_set": feature_set, "model": model_name, "params": params,
            "cv_macro_f1": f1_score(y["stacker"], to_labels(probs), average="macro"), "oof": probs}


cv_runs = [run_cv(config) for config in tqdm(configs, desc="CV")]
cv_results = pd.DataFrame(cv_runs).drop(columns="oof")

CV:   0%|          | 0/82 [00:00<?, ?it/s]

In [7]:
cv_results.assign(params=cv_results["params"].astype(str)).sort_values("cv_macro_f1", ascending=False).head(15).round(4)

,feature_set,model,params,cv_macro_f1
32,E2: Branch A + curves,HGB,"{'class_weight': 'balanced', 'learning_rate': ...",0.9397
30,E2: Branch A + curves,HGB,"{'class_weight': 'balanced', 'learning_rate': ...",0.9392
31,E2: Branch A + curves,HGB,"{'class_weight': 'balanced', 'learning_rate': ...",0.9391
26,E2: Branch A + curves,HGB,"{'class_weight': None, 'learning_rate': 0.05, ...",0.9390
27,E2: Branch A + curves,HGB,"{'class_weight': None, 'learning_rate': 0.05, ...",0.9387
28,E2: Branch A + curves,HGB,"{'class_weight': None, 'learning_rate': 0.1, '...",0.9384
29,E2: Branch A + curves,HGB,"{'class_weight': None, 'learning_rate': 0.1, '...",0.9383
33,E2: Branch A + curves,HGB,"{'class_weight': 'balanced', 'learning_rate': ...",0.9378
23,E2: Branch A + curves,LogReg,"{'C': 1.0, 'class_weight': 'balanced', 'max_it...",0.9361
21,E2: Branch A + curves,LogReg,"{'C': 0.1, 'class_weight': 'balanced', 'max_it...",0.9360


In [8]:
WEIGHT_GRID = np.round(np.arange(0.5, 2.01, 0.1), 2)  # multipliers for human and fusion (ai fixed at 1)


def apply_weights(probs, weights):
    """Labels after multiplying the human and fusion probabilities by weights = (w_human, w_fusion)."""
    w_human, w_fusion = weights
    return to_labels(np.asarray(probs) * np.array([w_human, 1.0, w_fusion]))


def tune_weights(probs, y_true):
    """Class weights from WEIGHT_GRID that maximise macro-F1 of the weighted argmax."""
    candidates = [(w_human, w_fusion) for w_human in WEIGHT_GRID for w_fusion in WEIGHT_GRID]
    return max(candidates, key=lambda w: f1_score(y_true, apply_weights(probs, w), average="macro"))


def fast_macro_f1(y_true, y_pred):
    """Macro-F1 from a 3 x 3 confusion matrix of class indices (for the bootstrap)."""
    confusion = np.bincount(3 * y_true + y_pred, minlength=9).reshape(3, 3)
    tp = np.diag(confusion)
    return np.mean(2 * tp / np.maximum(confusion.sum(axis=0) + confusion.sum(axis=1), 1))


def bootstrap_diff(y_true, pred, pred_ref, n=1_000):
    """95% paired-bootstrap interval of macro-F1(pred) - macro-F1(pred_ref) over resampled texts."""
    index = {label: i for i, label in enumerate(LABELS)}
    t, p, r = (np.vectorize(index.get)(a) for a in (y_true, pred, pred_ref))
    rng = np.random.default_rng(SEED)
    diffs = [fast_macro_f1(t[s], p[s]) - fast_macro_f1(t[s], r[s])
             for s in (rng.integers(0, len(t), len(t)) for _ in range(n))]
    return np.percentile(diffs, [2.5, 97.5])

In [9]:
def best_per_feature_set(cv_runs):
    """The configuration with the highest CV macro-F1 for each feature set."""
    best = {}
    for run in cv_runs:
        if run["feature_set"] not in best or run["cv_macro_f1"] > best[run["feature_set"]]["cv_macro_f1"]:
            best[run["feature_set"]] = run
    return best


def evaluate(run):
    """Refit the chosen configuration on the stacker set and predict validation with tuned class weights."""
    columns = columns_for(FEATURE_SETS[run["feature_set"]], X["stacker"].columns)
    weights = tune_weights(run["oof"], y["stacker"])
    model = make_model(run["model"], run["params"]).fit(X["stacker"][columns], y["stacker"])
    classes = list(model.classes_)
    val_probs = model.predict_proba(X["val"][columns])[:, [classes.index(label) for label in LABELS]]
    return {"model": model, "columns": columns, "weights": weights, "val_probs": val_probs,
            "val_pred": apply_weights(val_probs, weights)}


R0_PRED = to_labels(probs_a["val"])
r1_weights = tune_weights(probs_a["stacker"].to_numpy(), y["stacker"])
references = {"R0: Branch A, argmax": {"val_pred": R0_PRED, "weights": None},
              "R1: Branch A + class weights": {"val_pred": apply_weights(probs_a["val"], r1_weights), "weights": r1_weights}}
best_runs = best_per_feature_set(cv_runs)
fitted = {name: evaluate(run) for name, run in tqdm(best_runs.items(), desc="refit + val")}

refit + val:   0%|          | 0/11 [00:00<?, ?it/s]

In [10]:
def results_table(references, best_runs, fitted):
    """One row per configuration: CV score, validation scores and the bootstrap interval of the gain over R0."""
    rows = []
    for name, ref in references.items():
        rows.append({"configuration": name, "model": "-", "cv_macro_f1": np.nan, "weights": ref["weights"],
                     **score(y["val"], ref["val_pred"]), "ci_low": np.nan, "ci_high": np.nan})
    for name, fit in fitted.items():
        low, high = bootstrap_diff(y["val"], fit["val_pred"], R0_PRED)
        run = best_runs[name]
        rows.append({"configuration": name, "model": f"{run['model']} {run['params']}",
                     "cv_macro_f1": run["cv_macro_f1"], "weights": fit["weights"],
                     **score(y["val"], fit["val_pred"]), "ci_low": low, "ci_high": high})
    table = pd.DataFrame(rows).set_index("configuration")
    table.insert(table.columns.get_loc("macro_f1") + 1, "gain_vs_R0", table["macro_f1"] - table.loc["R0: Branch A, argmax", "macro_f1"])
    return table


results = results_table(references, best_runs, fitted)

In [11]:
results.drop(columns=["model", "weights"]).round(4)

,cv_macro_f1,macro_f1,gain_vs_R0,f1_human,f1_ai,f1_fusion,ci_low,ci_high
configuration,,,,,,,,
"R0: Branch A, argmax",NaN,0.9023,0.0000,0.8368,0.9901,0.8801,NaN,NaN
R1: Branch A + class weights,NaN,0.9085,0.0061,0.8539,0.9912,0.8803,NaN,NaN
R2: Branch A,0.9123,0.9081,0.0057,0.8544,0.9916,0.8783,0.0028,0.0086
E1: curves only,0.8961,0.8956,-0.0068,0.8503,0.9882,0.8483,-0.0113,-0.0022
E2: Branch A + curves,0.9397,0.9344,0.0320,0.9003,0.9933,0.9095,0.0282,0.0359
E3: Branch A + 64,0.9296,0.9256,0.0233,0.8850,0.9936,0.8983,0.0196,0.0269
E3: Branch A + 128,0.9321,0.9292,0.0269,0.8916,0.9935,0.9027,0.0233,0.0305
E3: Branch A + 256,0.9306,0.9266,0.0242,0.8885,0.9934,0.8979,0.0202,0.0279
"E3: Branch A + 128, 256",0.9351,0.9322,0.0299,0.8967,0.9936,0.9064,0.0260,0.0336
